# GoPro 屏幕提取器
**用法：** 点击顶部菜单 **Runtime → Run all**，等待最后一格出现链接，点击即可使用。无需注册账号。

In [ ]:
!pip install -q gradio opencv-python-headless Pillow gdown

In [ ]:
import cv2
import numpy as np
import tempfile, os, re
import gdown

# ── Google Drive download ─────────────────────────────────────────────

def download_from_drive(url_or_id: str) -> str:
    """Download a Google Drive file and return local path."""
    # Extract file ID from various Drive URL formats
    m = re.search(r'/d/([a-zA-Z0-9_-]{20,})', url_or_id)
    file_id = m.group(1) if m else url_or_id.strip()
    out = tempfile.mktemp(suffix='.mp4')
    gdown.download(id=file_id, output=out, quiet=False, fuzzy=True)
    if not os.path.exists(out) or os.path.getsize(out) < 1000:
        raise ValueError('Google Drive 下载失败。请确认文件已设为「任何人可查看」。')
    return out

# ── helpers ───────────────────────────────────────────────────────────

def enhance_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    frame = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(frame, (0,0), 3)
    return cv2.addWeighted(frame, 1.5, blur, -0.5, 0)

def order_points(pts):
    rect = np.zeros((4,2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)];  rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect

def detect_screen(frame, scale=0.5):
    small = cv2.resize(frame, None, fx=scale, fy=scale)
    gray  = cv2.bilateralFilter(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), 9, 75, 75)
    edges = cv2.dilate(cv2.Canny(gray, 30, 100),
                       cv2.getStructuringElement(cv2.MORPH_RECT,(3,3)), iterations=2)
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
    h, w = small.shape[:2]
    best, best_area = None, 0
    for c in cnts[:15]:
        area = cv2.contourArea(c)
        if not (h*w*0.05 < area < h*w*0.95): continue
        approx = cv2.approxPolyDP(c, 0.02*cv2.arcLength(c,True), True)
        if len(approx) == 4:
            pts = order_points(approx.reshape(4,2).astype('float32'))
            ww = np.linalg.norm(pts[1]-pts[0]); hh = np.linalg.norm(pts[3]-pts[0])
            if hh and 0.5 < ww/hh < 3.0 and area > best_area:
                best, best_area = pts / scale, area
    return best

def out_size(pts):
    tl,tr,br,bl = pts
    W = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
    H = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))
    return (W//2)*2, (H//2)*2

def warp(frame, pts, W, H):
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype='float32')
    return cv2.warpPerspective(frame, cv2.getPerspectiveTransform(pts, dst), (W,H))

def flow_track(pg, cg, pts):
    nxt, st, _ = cv2.calcOpticalFlowPyrLK(
        pg, cg, pts.reshape(4,1,2).astype('float32'), None,
        winSize=(31,31), maxLevel=4,
        criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))
    return nxt.reshape(4,2) if st is not None and st.sum()==4 else None

def ema(history, new, a=0.3):
    return new if not history else history[-1]*(1-a) + new*a

# ── main pipeline ─────────────────────────────────────────────────────

def process_video(input_path, output_path, progress_cb=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened(): raise ValueError('无法打开视频文件')
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    init_pts = None
    for _ in range(min(120, total)):
        ret, f = cap.read()
        if not ret: break
        init_pts = detect_screen(f)
        if init_pts is not None: break
    if init_pts is None:
        cap.release()
        raise ValueError('未能检测到屏幕矩形。请确认 GoPro 屏幕在视频开头清晰可见。')
    W, H = out_size(init_pts)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W,H))
    cur_pts = init_pts.copy()
    history = [cur_pts]
    prev_gray = None
    fails = 0
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            fp = flow_track(prev_gray, gray, cur_pts)
            if fp is not None:
                cur_pts = ema(history, fp);  history.append(cur_pts);  fails = 0
            else:
                fails += 1
        if prev_gray is None or fails >= 15:
            np_ = detect_screen(frame)
            if np_ is not None:
                cur_pts = ema(history, np_, 0.25);  history.append(cur_pts);  fails = 0
        prev_gray = gray
        try:
            writer.write(enhance_frame(warp(frame, cur_pts, W, H)))
        except Exception:
            writer.write(np.zeros((H,W,3), dtype=np.uint8))
        idx += 1
        if progress_cb and idx % 20 == 0:
            progress_cb(min(idx/total, 0.99), desc=f'处理中… {idx}/{total} 帧')
    cap.release();  writer.release()
    return output_path

print('处理逻辑加载完成 ✓')

In [ ]:
import gradio as gr, tempfile, traceback

def run(drive_url, video_file, progress=gr.Progress(track_tqdm=False)):
    tmp_drive = None
    try:
        if drive_url and drive_url.strip():
            progress(0.0, desc='从 Google Drive 下载中…')
            tmp_drive = download_from_drive(drive_url.strip())
            input_path = tmp_drive
        elif video_file:
            input_path = video_file
        else:
            raise gr.Error('请粘贴 Google Drive 链接，或直接上传视频文件')

        progress(0.05, desc='开始处理…')
        out = tempfile.mktemp(suffix='_extracted.mp4')
        process_video(input_path, out, progress_cb=progress)
        progress(1.0, desc='完成！')
        return out

    except gr.Error:
        raise
    except Exception as e:
        # Show full error so we can diagnose
        msg = f'{type(e).__name__}: {e}\n\n{traceback.format_exc()}'
        print(msg)
        raise gr.Error(msg)
    finally:
        if tmp_drive and os.path.exists(tmp_drive):
            os.remove(tmp_drive)

with gr.Blocks(title='GoPro 屏幕提取器', theme=gr.themes.Base()) as demo:
    gr.Markdown('# 🎥 GoPro 屏幕提取器')
    with gr.Row():
        with gr.Column():
            drive_url = gr.Textbox(
                label='方式一：Google Drive 链接（推荐，无需上传）',
                value='https://drive.google.com/file/d/1cZVlq95JMxtIaV8caaCsY6P-_q_VGx3p/view?usp=drivesdk',
                lines=1
            )
            gr.Markdown('**或者**')
            video_file = gr.Video(
                label='方式二：直接上传视频',
                sources=['upload']
            )
            btn = gr.Button('开始提取', variant='primary', size='lg')
        with gr.Column():
            vid_out = gr.Video(label='提取结果（完成后可下载）', interactive=False)
            err_box = gr.Textbox(label='错误详情（出错时显示）', visible=True, interactive=False, lines=6)
    btn.click(fn=run, inputs=[drive_url, video_file], outputs=vid_out)

demo.queue(max_size=2).launch(share=True, debug=True, show_error=True)